In [25]:
import pandas as pd
df=pd.read_csv('c:/data/boston/house.csv')
df

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,target,target2
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0,1
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6,0
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7,1
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4,1
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4,0
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6,0
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9,1
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0,0


In [26]:
X = df.iloc[:, :-2]
y= df.iloc[:, [-2]]

In [27]:
import numpy as np

N=len(df)
ratio=0.7
np.random.seed(0)
idx_train = np.random.choice(np.arange(N), np.int64(ratio*N), replace=False)
idx_test = list(set(np.arange(N)).difference(idx_train))
df_train = df.iloc[idx_train]
df_test = df.iloc[idx_test]

In [28]:
print(len(np.arange(N)))
print(np.int64(ratio*N))

506
354


In [31]:
import statsmodels.api as sm
model = sm.OLS.from_formula('target~' + '+'.join(X.columns), data=df_train)
result = model.fit()
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 target   R-squared:                       0.728
Model:                            OLS   Adj. R-squared:                  0.718
Method:                 Least Squares   F-statistic:                     70.06
Date:                Thu, 15 Jan 2026   Prob (F-statistic):           8.57e-88
Time:                        15:24:05   Log-Likelihood:                -1043.0
No. Observations:                 354   AIC:                             2114.
Df Residuals:                     340   BIC:                             2168.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     35.0719      5.932      5.913      0.000      23.404      46.739
CRIM          -0.1005      0.035     -2.869      0.004      -0.169      -0.032
ZN             0.0381      0.017      2.248      0.025       0.005       0.071
INDUS          0.0202      0.072      0.281      0.779      -0.121       0.162
CHAS           1.1498      1.020      1.127      0.260      -0.856       3.156
NOX          -17.3942      4.522     -3.847      0.000     -26.288      -8.500
RM             3.8640      0.485      7.959      0.000       2.909       4.819
AGE            0.0004      0.016      0.023      0.982      -0.030       0.031
DIS           -1.3285      0.236     -5.626      0.000      -1.793      -0.864
RAD            0.3741      0.084      4.447      0.000       0.209       0.540
TAX           -0.0160      0.005     -3.315      0.001      -0.025      -0.007
PTRATIO       -0.8989      0.153     -5.885      0.000      -1.199      -0.598
B              0.0095      0.003      3.015      0.003       0.003       0.016
LSTAT         -0.5013      0.060     -8.423      0.000      -0.618      -0.384
==============================================================================
Omnibus:                      136.641   Durbin-Watson:                   1.998
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              602.833
Skew:                           1.618   Prob(JB):                    1.25e-131
Kurtosis:                       8.513   Cond. No.                     1.47e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.47e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [32]:
pred = result.predict(df_test)
#Residual Sum of Square 잔차제곱합
rss = ((df_test.target - pred)**2).sum()
#Total Sum of Square 총제곱합
tss = ((df_test.target - df_test.target.mean())**2).sum()
rsquared = 1-rss/tss
rsquared

np.float64(0.7519796502601113)

In [33]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.3, random_state=0)
df_train.shape, df_test.shape


((354, 15), (152, 15))

In [34]:
dfX_train, dfX_test, dfy_train, dfy_test = train_test_split(X, y, test_size=0.3, random_state=0)
dfX_train.shape, dfX_test.shape, dfy_train.shape, dfy_test.shape

((354, 13), (152, 13), (354, 1), (152, 1))

In [41]:
from sklearn.model_selection import KFold
scores = np.zeros(7)
cv = KFold(7, shuffle=True, random_state=0)
for i, (idx_train, idx_test) in enumerate(cv.split(df)):
    df_train = df.iloc[idx_train]
    df_test = df.iloc[idx_test]
    model = sm.OLS.from_formula('target~' + '+'.join(X.columns), data=df_train)
    result = model.fit()
    pred = result.predict(df_test)
    rss = ((df_test.target - pred)**2).sum()
    tss = ((df_test.target - df_test.target.mean())**2).sum()
    rsquared = 1-rss/tss
    scores[i] = rsquared
    print(f'학습용 R2 = {result.rsquared:.3f}, 검증용 R2 = {rsquared:.3f}')

학습용 R2 = 0.767, 검증용 R2 = 0.535
학습용 R2 = 0.736, 검증용 R2 = 0.758
학습용 R2 = 0.738, 검증용 R2 = 0.740
학습용 R2 = 0.754, 검증용 R2 = 0.587
학습용 R2 = 0.732, 검증용 R2 = 0.784
학습용 R2 = 0.742, 검증용 R2 = 0.726
학습용 R2 = 0.727, 검증용 R2 = 0.808


In [36]:
np.ones(5)

array([1., 1., 1., 1., 1.])

In [37]:
a = [10,20,30,40,50]
for i,val in enumerate(a):
    print(i,val)

0 10
1 20
2 30
3 40
4 50


In [38]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
scores1 = np.zeros(5)
scores2 = np.zeros(5)
scores3 = np.zeros(5)
cv = KFold(5,shuffle=True, random_state=0)
for i, (idx_train, idx_test) in enumerate(cv.split(df)):
    df_train = df.iloc[idx_train]
    df_test = df.iloc[idx_test]
    model = sm.OLS.from_formula('target~' + '+'.join(X.columns), data=df_train)
    result = model.fit()
    pred = result.predict(df_test)
    rsquared = r2_score(df_test.target, pred)
    scores1[i] = rsquared
    mse = mean_squared_error(df_test.target, pred)
    scores2[i] = mse
    mae = mean_absolute_error(df_test.target, pred)
    scores3[i] = mae

print(scores1)
print(scores2)
print(scores3)
print(np.mean(scores1))
print(np.mean(scores2))
print(np.mean(scores3))

[0.58922238 0.77799144 0.66791979 0.6680163  0.83953317]
[33.44898    18.65881615 21.23463289 29.22251557 16.57369039]
[3.84290922 3.38979394 3.07473854 3.6463452  3.03058651]
0.7085366175182795
23.82772699990081
3.396874681157454
